# Lesson 1: Simple ReAct Agent from Scratch

#### based -https://til.simonwillison.net/llms/python-react-pattern

It is advised to pay attention of where the llm is used and when the python code is exectured that is created by the user. 

**Opinion - Great lesson**

Below a simple agent is coded from scratch using llm and some local tools. This allows a deeper understanding of how an agents works. The wonder is to create a very well crafted ReAct prompt. It was an eye opener to me!

In [ ]:
import sys
import os
import openai

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key, tavily_api_key

#openai.api_key = api_key
os.environ['OPENAI_API_KEY'] = api_key
os.environ['TAVILY_API_KEY'] = tavily_api_key

In [ ]:
import openai
from openai import OpenAI
import re
import httpx
import os

In [ ]:
client = OpenAI()

In [ ]:
chat_completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello World."}]
)

In [ ]:
chat_completion.choices[0].message.content

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        # create a system role a first message if a system message is
        # provided
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        # first step is to append the message to the list of messages
        # in a format that openai .chat.completions requries
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [ ]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [ ]:
abot = Agent(prompt)
abot.messages

In [ ]:
result = abot("How much does a toy poodle weigh?")
print(result)

In [ ]:
result = average_dog_weight("Toy Poodle")

In [ ]:
abot.messages

In [ ]:
next_prompt = "Observation: {}".format(result)
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
abot.messages

Let's do a more complex task. For this we need a new agent without historical messages

In [ ]:
abot = Agent(prompt)

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
abot.messages

In [ ]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
abot.messages

### Add loop 

The code snippet below shows intermediate steps of how the result of a call to the Agent can can be used to select and action and execute real python code.  

In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

# extract an example result from above
example_result = abot.messages[4]['content']
display(example_result)

# split the string and only take the first element
actions = action_re.match(abot.messages[4]['content'].split('\n')[0])
print(actions)

action, action_input = actions.groups()
print(f"The function to call: {action}")
print(f"Input argument to the function: {action_input}")

# use `action` to select the correct function from a dictionary 
observation = known_actions[action](action_input)
print(f"Observation: {observation}")

> Note: I added some print statements for verbose output to the function below

In [ ]:
def query(question, max_turns=5):
    i = 0
    # initiate the agent with the ReAct role prompt
    bot = Agent(prompt)
    # set next prompt to the question
    next_prompt = question
    while i < max_turns:
        i += 1
        # invoke the __call__ function of the agent with the next prompt
        result = bot(next_prompt)
        print(result)
        # create a list of actions 
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        print(actions)

        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question, max_turns=6)

# Lesson 2 : LangGraph Components

The visual at the start of the second lesson really helps transition from Lesson 1 to langchain. 

> **Note 1** - The related video is in my view rather complex on things like `operator.add` and exactly how the state is persisted or overwritten in certain cases.
> **Note 2** - The objective of this lesson (maybe more lectures) is how the basically adjust the Agent from Lesson 1 by langchain components. It is **not** how you would actually build an agent using langchain. However more experience is required to fully understand.

In [ ]:
from typing import TypedDict, Annotated
import operator

# langchain standard imports
from langchain_core.messages import AIMessage, AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

# imports for building an agen
from langgraph.graph import StateGraph, END

# langchain tools
from langchain_tavily import TavilySearch
from langchain_experimental.tools import PythonREPLTool

Below a short overview of some langchain messages (`HumanMessage`, `SystemMessage`, `AIMessage`) and how these relate to the openai chat.completions `messages`. The messages created using langchain messages can be stored in a list. The messages can be given as input using the langchain `.invoke` methods. The output is normally an `AImessaga` that can be appended to messages. By appending a new `HumanMessage` a full history can be build and provided in subsequent calls (maybe my explenation is not that clear but you get the point)

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini')

# System message sets context / behavior
system_msg = SystemMessage(content="You are a helpful assistant.")
# Equivalent OpenAI Chat API format:
# {"role": "system", "content": "You are a helpful assistant."}

# Human message is the user's input
human_msg = HumanMessage(content="Hello, can you tell me a joke?")
# Equivalent OpenAI Chat API format:
# {"role": "user", "content": "Hello, can you tell me a joke?"}

# Optional: if you want to include a prior assistant message
assistant_msg = AIMessage(content="Sure! Here's one: Why did the chicken cross the road?")
# Equivalent OpenAI Chat API format:
# {"role": "assistant", "content": "Sure! Here's one: Why did the chicken cross the road?"}

messages = [system_msg, human_msg, assistant_msg]

# start invokation with system and human message
result = llm.invoke(messages)
#type(result)

#print(result.content)

# add the response to the end
# messages has system, human and ai message
messages.append(result)
# #print(messages)

# add human message
messages.append(HumanMessage(content="Tell me another one about a cat?"))
result = llm.invoke(messages)

# add ai message
# messages has : system, human, ai, human, ai
messages.append(result)

for msg in messages:
    print(type(msg))
    print(msg.content)
    print()

In [ ]:
# This will later be used to check if a tool has been called
print(messages[-1])
print(messages[-1].tool_calls)
print(len(messages[-1].tool_calls))

The above helps understanding the code below of the `Agent` class

In [ ]:
tool = TavilySearch(max_results=2,api_key=tavily_api_key) #increased number of results
print(type(tool))
print(tool.name)

repl_tool = PythonREPLTool()
print(type(repl_tool))
print(repl_tool.name)

In [ ]:
# System message sets context / behavior
system_msg = SystemMessage(content="You are a helpful assistant. That can call two tools to answer user questions in case these are relevant."
                          "The available tools are tavily_search and Python_REPL. The Python_REPL tool can be used to write code when requested.")
    
# Human message is the user's input
human_msg = HumanMessage(content="Write a small working programm that contains a function to add 2 numbers?")
# Equivalent OpenAI Chat API format:
# {"role": "user", "content": "Write a small working programm that contains a function to add 2 numbers?"}

# create messages list
messages = [system_msg, human_msg]

# create a tools list
tools = [tool, repl_tool]

# create an langchain llm object and bind the tools
llm = ChatOpenAI(model='gpt-4o-mini')
llm_with_tools = llm.bind_tools(tools=tools)

# invoke the llm with tools
result = llm_with_tools.invoke(input=messages)

In [ ]:
# later on the below is ued to check if a tool was called or not
print(result.tool_calls[0])

# The result.tool_calls returns a list as several functions might be called in parallel
print(type(result.tool_calls))

> If you are not familiar with python typing annotation, you can refer to the [python documents](https://docs.python.org/3/library/typing.html).

Below we create an `AgentState` of `TypeDict` with a key `messages` that is an annotated list of type `AnyMessage`. Wow that is a mouth full.

This will basically be the memory of the agent that adds messages to object of class AgentState.

In [ ]:
from langgraph.graph.message import add_messages

# Option 1
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# Option 2
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

> **Note** - in the above code snippet you must use `operator.add` or `add_messages`. This is for the internal working of langchain that it appends the messages to create a history. If you would code the agent yourself you would not need this (`state['messages'].append(new_message)`)

Below we show a simple example of adding messages to the object using `state['messages']`

In [ ]:
state = AgentState()

# add add a message to the messages key
state["messages"] = [{"role" : "system", "content": "You are a helpful assistent"}]
print(state)  # {"messages": ["hello world"]}

state['messages'].append({"role": "user", "content": "hello world"})
print(state)
print(type(state))

llm.invoke(state['messages']).content

> **Note** - in `take_action` below, some logic was added to cover the case that the LLM returned a non-existent tool name. Even with function calling, LLMs can still occasionally hallucinate. Note that all that is done is instructing the LLM to try again! An advantage of an agentic organization.

> **Note** - my understanding Agent below -> Each node has an related function. The `llm` nodel calls the `call_openai` method and the `action` node calls the `take_action` method. The `add_conditional_edge` is a router to a node based on the result of the `llm` node result. It can route to the `END` note if there is no function call OR it can route to the `action` node in case a tool call is required. Any result of the `action` node is routed back to the `llm` node for a `final` llm call.

> **Note** - I added some print statements to better understand the flow in the agent 

In [ ]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system

        # my understanding is that we build the agent using graphs
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()  # I think this is now the Agent
        
        # create a dictionary from tools {name: tool}
        # and bind the tools to the model
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        # if no tool was called len(results.tool_calls)=0
        # then False is returned else if a tool was called True
        is_tool_called : Bool = len(result.tool_calls) > 0

        if is_tool_called:
            print("Going to the action node")
        else:
            print("Going to the END node")
        return is_tool_called

    def call_openai(self, state: AgentState):
        # messages is a list of different message types (human, systen, ai)
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        print("Calling the llm")
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        # This node is called only IF a tool is called by the llm
        # tool_calls is a list
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                # self.tools is a dictionary with tool name as key and the function as value
                # tools[t['name']] returns the function
                result = self.tools[t['name']].invoke(t['args'])
                # add the result to messages
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOpenAI(model="gpt-3.5-turbo")  #reduce inference cost

# Create the agent
# remark - I added a second tool myself to play with the code
abot = Agent(model, [tool, repl_tool], system=prompt)

In [ ]:
abot.graph

In [ ]:
abot.tools['tavily_search']

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]
result = abot.graph.invoke({"messages": messages})

In [ ]:
for i,msg in enumerate(result['messages']):
    print(i)
    print(msg)
    print()

In [ ]:
result['messages'][1].tool_calls

In [ ]:
result['messages'][-1].content

In [ ]:
messages = [HumanMessage(content="What is the weather in Voorchoten in the Netherlands?")]
result = abot.graph.invoke({"messages": messages})

In [ ]:
result['messages'][-1].content

In [ ]:
# Results may vary per run and over time as search information and models change.

query = "Who won the super bowl in 2024? In what state is the winning team headquarters located? \
What is the GDP of that state? Answer each question." 
messages = [HumanMessage(content=query)]

model = ChatOpenAI(model="gpt-4o")  # requires more advanced model
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})

In [ ]:
print(result['messages'][-1].content)

# Lesson 3: Agentic Search

In [ ]:
import os
from tavily import TavilyClient

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

# run search
result = client.search("What is in Nvidia's new Blackwell GPU?", include_answer=True)

# print the answer
result["answer"]

## Regular search

In [ ]:
# choose location (try to change to your own city!)

city = "Voorschoten"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [ ]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

result = ddg.text(query, max_results=2)
print(result)

In [ ]:
from ddgs import DDGS

def search(query, max_results=2):
    with DDGS() as ddg:
        return list(ddg.text(query, max_results=max_results))

# call function
result = search(query)

# this search returns a list of dictionaries where each
# dictionary has a reference link that can scraped for 
# more information
[i['href'] for i in result]

In [ ]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

The `scrape_weather` function below downloads the website `url`.

In [ ]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [ ]:
# use DuckDuckGo to find websites and take the first result
urls = search(query)
print(urls)    
url = urls[1]
print(url)

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:500]) # limit long outputs

`BeautifulSoup` is a popular package to parse specific data from a website. (Of course nowadays llms can be used to parse data from websites)


In [ ]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)

print(f"Website: {url}\n\n")
print(weather_data)

## Agentic Search

In [ ]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

In [ ]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


# Lesson 4: Persistence and Streaming

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

In [ ]:
tool = TavilySearch(max_results=2, api_key=tavily_api_key)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

> **Note** - Below are two possible approaches to work with memory one `InMemorySaver` and `SqliteSaver`. 

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver
import sqlite3

# This does work
#memory = InMemorySaver()

# This should create the memory in an sqlite database.
memory = SqliteSaver(sqlite3.connect("checkpoint.db",check_same_thread=False))

The earlier `Agent` has a new argument `checkpointer`

In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        
        # HERE the checkpointer is added
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [ ]:
config = {
    "configurable": {"thread_id": "1"}
}

In [ ]:
abot.graph.invoke({"messages":messages}, config)

if you would restart the kernel and NOT run the cell above. You can still get access to the saved memory from the sqlite database.

In [ ]:
# config = {"configurable": {"thread_id": "1"}}
# abot.graph.get_state(config)

In [ ]:
for event in abot.graph.stream({"messages": messages}, config=config):
    for v in event.values():
        print(v['messages'])

In [ ]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
# uncomment the line below to see the total history of thread 1
# abot.graph.get_state(config)

below the `thread` has been changed from 1 to 2. `thread` 2 has no data stored So the agent should be confused. 

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

## Streaming tokens

In [ ]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
import aiosqlite

# This should create the memory in an sqlite database.
memory = AsyncSqliteSaver(aiosqlite.connect("async_checkpoint.db",check_same_thread=False))
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

> **Note** - I did not get the same results as in the deeplearning.ai course

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

# Lesson 5: Human in the Loop

Note: This notebook is running in a later version of langgraph that it was filmed with. The later version has a couple of key additions:
- Additional state information is stored to memory and displayed when using `get_state()` or `get_state_history()`.
- State is additionally stored every state transition while previously it was stored at an interrupt or at the end.
These change the command output slightly, but are a useful addtion to the information available.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver
import sqlite3

# This does work
#memory = InMemorySaver()

# This should create the memory in an sqlite database.
memory = SqliteSaver(sqlite3.connect(":memory:",check_same_thread=False))

In [ ]:
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # merge the new messages with the existing messages
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # append any new messages to the end
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [ ]:
from langchain_tavily import TavilySearch
tool = TavilySearch(max_results=2, api_key=tavily_api_key)

## Manual human approval

In [ ]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-3.5-turbo")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

It stops here because we have `interrupt_before=['action']`

In [ ]:
abot.graph.get_state(thread)

In [ ]:
abot.graph.get_state(thread).next

### continue after interrupt

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread)

In [ ]:
abot.graph.get_state(thread).next

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

## Modify State
Run until the interrupt and then modify the state.

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread)

In [ ]:
current_values = abot.graph.get_state(thread)

In [ ]:
current_values.values['messages'][-1]

In [ ]:
current_values.values['messages'][-1].tool_calls

In [ ]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search',
  'args': {'query': 'current weather in Louisiana'},
  'id': _id}
]

In [ ]:
abot.graph.update_state(thread, current_values.values)

In [ ]:
abot.graph.get_state(thread)

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

## Time Travel

In [ ]:
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)

To fetch the same state as was filmed, the offset below is changed to `-3` from `-1`. This accounts for the initial state `__start__` and the first state that are now stored to state memory with the latest version of software.

In [ ]:
to_replay = states[-3]

In [ ]:
to_replay.values['messages']

starting from `to_replay`

In [ ]:
for event in abot.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)

## Go back in time and edit

In [ ]:
to_replay

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [ ]:
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

In [ ]:
for event in abot.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)

## Add message to a state at a given time

In [ ]:
to_replay

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']

In [ ]:
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="tavily_search",
    content="54 degree celcius",
)]}

In [ ]:
branch_and_add = abot.graph.update_state(
    to_replay.config, 
    state_update, 
    as_node="action")

In [ ]:
for event in abot.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)

# Extra Practice

## Build a small graph
This is a small simple graph you can tinker with if you want more insight into controlling state memory.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

Define a simple 2 node graph with the following state:
-`lnode`: last node
-`scratch`: a scratchpad location
-`count` : a counter that is incremented each step

In [ ]:
# This should create the memory in an sqlite database.
memory = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False))

# memory = SqliteSaver.from_conn_string(":memory:")
graph = builder.compile(checkpointer=memory)

In [ ]:
class AgentState(TypedDict):
    lnode: str
    scratch: str
    count: Annotated[int, operator.add]

In [ ]:
def node1(state: AgentState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1",
            "count": 1,
           }
def node2(state: AgentState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2",
            "count": 1,
           }

The graph goes N1->N2->N1... but breaks after count reaches 3.

In [ ]:
def should_continue(state):
    return state["count"] < 3

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)

builder.add_edge("Node1", "Node2")
builder.add_conditional_edges("Node2", 
                              should_continue, 
                              {True: "Node1", False: END})
builder.set_entry_point("Node1")

### Run it!
Now, set the thread and run!

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
graph.invoke({"count":0, "scratch":"hi"},thread)

### Look at current state

Get the current state. Note the `values` which are the AgentState. Note the `config` and the `thread_ts`. You will be using those to refer to snapshots below.

In [ ]:
graph.get_state(thread)

View all the statesnapshots in memory. You can use the displayed `count` agentstate variable to help track what you see. Notice the most recent snapshots are returned by the iterator first. Also note that there is a handy `step` variable in the metadata that counts the number of steps in the graph execution. This is a bit detailed - but you can also notice that the *parent_config* is the *config* of the previous node. At initial startup, additional states are inserted into memory to create a parent. This is something to check when you branch or *time travel* below.

### Look at state history

In [ ]:
for state in graph.get_state_history(thread):
    print(state, "\n")

Store just the config into an list. Note the sequence of counts on the right. get_state_history returns the most recent snapshots first.

In [ ]:
states = []
for state in graph.get_state_history(thread):
    states.append(state.config)
    print(state.config, state.values['count'])

Grab an early state.

In [ ]:
states[-3]

This is the state after Node1 completed for the first time. Note `next` is `Node2`and `count` is 1.

In [ ]:
graph.get_state(states[-3])

### Go Back in Time
Use that state in `invoke` to go back in time. Notice it uses states[-3] as *current_state* and continues to node2,

In [ ]:
graph.invoke(None, states[-3])

Notice the new states are now in state history. Notice the counts on the far right.

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state.config, state.values['count'])

You can see the details below. Lots of text, but try to find the node that start the new branch. Notice the parent *config* is not the previous entry in the stack, but is the entry from state[-3].

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state,"\n")

### Modify State
Let's start by starting a fresh thread and running to clean out history.

In [ ]:
thread2 = {"configurable": {"thread_id": str(2)}}
graph.invoke({"count":0, "scratch":"hi"},thread2)

In [ ]:
graph

In [ ]:
states2 = []
for state in graph.get_state_history(thread2):
    states2.append(state.config)
    print(state.config, state.values['count'])   

Start by grabbing a state.

In [ ]:
save_state = graph.get_state(states2[-3])
save_state

Now modify the values. One subtle item to note: Recall when agent state was defined, `count` used `operator.add` to indicate that values are *added* to the current value. Here, `-3` will be added to the current count value rather than replace it.

In [ ]:
save_state.values["count"] = -3
save_state.values["scratch"] = "hello"
save_state

Now update the state. This creates a new entry at the *top*, or *latest* entry in memory. This will become the current state.

In [ ]:
graph.update_state(thread2,save_state.values)

Current state is at the top. You can match the `thread_ts`.
Notice the `parent_config`, `thread_ts` of the new node - it is the previous node.

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

# Lesson 6: Essay Writer

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage
import sqlite3

# This should create the memory in an sqlite database.
memory = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False))

In [ ]:
class AgentState(TypedDict):
    task: str
    plan: str            # plan from planning agent
    draft: str           # draft of the essay
    critique: str        # critique agent
    content: List[str]   # list of document travily
    revision_number: int
    max_revisions: int 

In [ ]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""

In [ ]:
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
be used when writing the following essay. Generate a list of search queries that will gather \
any relevant information. Only generate 3 queries max."""


In [ ]:
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed: 

------

{content}"""

In [ ]:
REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""

In [ ]:
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
be used when making any requested revisions (as outlined below). \
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


In [ ]:
from pydantic import BaseModel

class Queries(BaseModel):
    queries: List[str]

In [ ]:
from tavily import TavilyClient
import os
tavily = TavilyClient(api_key=tavily_api_key)

In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT), 
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    # updates the `plan` key with this response
    return {"plan": response.content}

In [ ]:
def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [ ]:
def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
        ]
    response = model.invoke(messages)
    return {
        "draft": response.content, 
        "revision_number": state.get("revision_number", 1) + 1
    }


In [ ]:
def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT), 
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

In [ ]:
def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [ ]:
def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

In [ ]:
builder = StateGraph(AgentState)

In [ ]:
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

In [ ]:
# graph starts at the planner
builder.set_entry_point("planner")

In [ ]:
builder.add_conditional_edges(
    "generate", 
    should_continue, 
    {END: END, "reflect": "reflect"}
)

In [ ]:
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")

builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

In [ ]:
graph = builder.compile(checkpointer=memory)

In [ ]:
graph

In [ ]:
thread = {"configurable": {"thread_id": "1"}}
for s in graph.stream({
    'task': "what is the difference between langchain and langsmith",
    'content': [],
    "max_revisions": 2,
    "revision_number": 1,
}, thread):
    print(s)

In [ ]:
print(graph.get_state(thread).values['draft'])

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from helper import ewriter, writer_gui

In [ ]:
MultiAgent = ewriter()
app = writer_gui(MultiAgent.graph)
app.launch()